限制模型调用次数，避免无限循环，控制调用成本。

## 举例1：会话限制

同一个会话线程，三次调用，每次调用请求一次模型

通过exit_behavior参数，可以控制达到限制后的行为。
默认值为"end"，即直接退出。
也可以设置为"error"，即抛出错误。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)



model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

from typing import List

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,  # 每个线程最多2次模型调用
            # run_limit=5,  # 每次运行最多5次
            exit_behavior="end",  # 达到限制后直接退出
            #exit_behavior="error",  # 达到限制后抛出错误
        ),
    ],
)

def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage | AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()

config = {"configurable": {"thread_id": "1"}} 
response_first = agent.invoke({
    "messages": [HumanMessage("你好")]
}, config=config)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])

response_second = agent.invoke({
    "messages": [HumanMessage("你是谁？")]
}, config=config)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])

response_third = agent.invoke({
    "messages": [HumanMessage("你能帮我做什么？")]
}, config=config)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是一个由 OpenAI 训练的人工智能助手。  
你可以把我当作一个能帮你回答问题、写作、翻译、总结、编程、头脑风暴的助手。

如果你愿意，我也可以直接进入状态帮你处理具体事情。
============================== > third < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==============================

## 举例2：运行限制

单次invoke最多调用5次模型，应对任务失败反复调用的情况。

需要fake-server重复触发工具调用

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv

load_dotenv()
model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            # thread_limit=2,
            run_limit=3,
            exit_behavior="end", # 达到限制后直接退出，也可以设置为"error"，即抛出错误
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("你好")]
}, config=config)

for msg in response["messages"]:
    msg.pretty_print()
